# Edge Deployment – Core ML Export Notebook

This notebook prepares a trained multi-head MobileNetV2 model for edge deployment on iOS devices.  
The previously trained PyTorch model (age, gender, and emotion classification) is reconstructed, its learned weights are loaded from a saved checkpoint, and the model is exported using TorchScript.

The TorchScript model is converted into a Core ML .mlpackage using coremltools, enabling efficient on-device inference on iPhone and iPad without requiring server-side computation.


## Imports

In [1]:
from google.colab import drive
import torch
from torch import nn
from torchvision import models
import os

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
!pip -q install coremltools==8.3.0
!pip -q install torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/2.3 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━ 1.0/2.3 MB 34.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━ 1.0/2.3 MB 34.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 2.3/2.3 MB 23.7 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/70.7 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 7.2 MB/s eta 0:00:00


In [4]:
device = torch.device("cpu")
print("Device:", device)

Device: cpu


## Recreate training model class



In [5]:
class AgeGenderEmotionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.mobilenet_v2(pretrained=False)  # IMPORTANT: pretrained False here
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()

        self.age_head = nn.Linear(in_features, 3)
        self.gender_head = nn.Linear(in_features, 2)
        self.emotion_head = nn.Linear(in_features, 7)

    def forward(self, x):
        feats = self.backbone(x)
        age_logits = self.age_head(feats)
        gender_logits = self.gender_head(feats)
        emotion_logits = self.emotion_head(feats)
        return age_logits, gender_logits, emotion_logits


## Load saved weights from Google Drive

In [6]:
model_path = "/content/drive/MyDrive/my_final_model.pth"
assert os.path.exists(model_path), f"Not found: {model_path}"

model = AgeGenderEmotionNet().to(device)
state = torch.load(model_path, map_location=device)
model.load_state_dict(state)
model.eval()

print("Loaded model weights OK")


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Loaded model weights OK


## Sanity check: runs one forward pass

In [7]:
example = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    age_logits, gender_logits, emotion_logits = model(example)

print(age_logits.shape, gender_logits.shape, emotion_logits.shape)


torch.Size([1, 3]) torch.Size([1, 2]) torch.Size([1, 7])


## TorchScript export

In [8]:
traced = torch.jit.trace(model, example)
ts_path = "age_gender_emotion.ts"
traced.save(ts_path)

print("Saved TorchScript:", ts_path)


Saved TorchScript: age_gender_emotion.ts


## Convert TorchScript to Core ML

In [ ]:
import coremltools as ct

mlmodel = ct.convert(
    traced,  # TorchScript model to convert (exported from PyTorch)
    inputs=[
        # Declares the model's input signature for Core ML
        ct.TensorType(name="image", shape=(1, 3, 224, 224))
    ],
    compute_units=ct.ComputeUnit.ALL,
)

out_path = "AgeGenderEmotion.mlpackage"
mlmodel.save(out_path)

print("Saved Core ML model:", out_path)


Running MIL default pipeline:   0%|          | 0/89 [00:00<?, ? passes/s]/usr/local/lib/python3.12/dist-packages/coremltools/converters/mil/mil/passes/defs/preprocess.py:273: UserWarning: Output, '820', of the source model, has been renamed to 'var_820' in the Core ML model.
  warnings.warn(msg.format(var.name, new_name))
/usr/local/lib/python3.12/dist-packages/coremltools/converters/mil/mil/passes/defs/preprocess.py:273: UserWarning: Output, '823', of the source model, has been renamed to 'var_823' in the Core ML model.
  warnings.warn(msg.format(var.name, new_name))
/usr/local/lib/python3.12/dist-packages/coremltools/converters/mil/mil/passes/defs/preprocess.py:273: UserWarning: Output, '826', of the source model, has been renamed to 'var_826' in the Core ML model.
  warnings.warn(msg.format(var.name, new_name))
Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 130.90 passes/s]


Saved Core ML model: AgeGenderEmotion.mlpackage


## Download the .mlpackage

In [13]:
from google.colab import files
files.download("AgeGenderEmotion.mlpackage")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>